# ContextNet + NicheNet Workflow

This tutorial demonstrates how to create publication-ready Sankey diagrams from **ContextNet/NicheNet** cell-cell communication analysis results using ccflowR.

### What You'll Learn

1. Load NicheNet output data into ccflowR
2. Define consistent cell type colors
3. Create a basic Sankey plot
4. Customize your figure for publication (coloring, highlighting, ordering)
5. Export figures

For detailed parameter reference (every option with visual side-by-side comparisons), see the **Parameter Reference Guide**.

### How ccflowR Fits in the ContextNet Pipeline

    ┌─────────────────────┐     ┌─────────────────┐     ┌─────────────┐
    │ GRN Inference       │────>│ ContextNet      │────>│ ccflowR     │
    │ (Snakemake)         │     │ (R package)     │     │ (Sankey)    │
    │                     │     │                 │     │             │
    │ scRNA + scATAC      │     │ GRN + NicheNet  │     │ Visualize   │
    │ → TF-target GRN     │     │ → Ligand scores │     │ → Figures   │
    └─────────────────────┘     └─────────────────┘     └─────────────┘
                                    │
                                    v
                              sankey_river.csv

ccflowR visualizes the signaling flow as: **Sender Cell → Ligand → Receptor → Receiver Cell**

## Data Requirements

### Required Columns
| Column | Type | Description |
|--------|------|-------------|
| `sender` | character | Sending cell type name |
| `receiver` | character | Receiving cell type name |
| `ligand` | character | Ligand gene symbol |
| `receptor` | character | Receptor gene symbol |

### Optional Columns (for Flow-Conserving Mode)
| Column | Type | Description |
|--------|------|-------------|
| `scaled_activity` | numeric | Combined prioritization score (0-1) |
| `scaled_receptor_expression` | numeric | Receptor expression weight (0-1) |

When these columns are present, ccflowR uses **flow-conserving mode** where edge thickness reflects biological signal strength.

### Loading Your Own Data

    # Load your NicheNet/ContextNet output file
    data <- read_nichenet("path/to/your/sankey_river.csv")
    head(data)

In [ ]:
library(ccflowR)

## Step 1: Load Data

**Why start here?** Before any visualization, you need to verify that your data loaded correctly and contains the expected columns. Inspecting the data early catches formatting issues before they cause cryptic errors downstream.

In [ ]:
# Use example data for demonstration
ex <- example_communication()
data <- ex$data

# Inspect the data structure
head(data)

**Why summarize?** `summarize_data()` gives a quick overview of your network: how many interactions, cell types, ligands, and receptors. This is the fastest way to check that no data was lost during import and that column names mapped correctly.

In [ ]:
# Get a summary of the communication data
summarize_data(data)

You should see 15 interactions involving 5 cell types (Macrophage, T_cell, B_cell, Fibroblast, Endothelial) with 11 unique ligands and 11 unique receptors. Notice that some ligands (TGFB1, TNF, IL6) are produced by multiple senders, and some receptors (TGFBR1, TNFRSF1A, IL6R, IFNGR1) appear on multiple receivers — these shared nodes create convergence and divergence patterns in the Sankey diagram. The data includes `scaled_activity` and `scaled_receptor_expression` columns, which enable flow-conserving mode.

## Step 2: Define Cell Type Colors

**Why define colors explicitly?** When generating multiple figures for a paper or presentation, using the same color for each cell type across all panels makes it easier for readers to track cell types at a glance.

In [ ]:
# Use colors from the example data
celltype_colors <- ex$colors

# View the color assignments
print(celltype_colors)

# Verify all cell types in data have colors assigned
all_celltypes <- get_all_celltypes(data)
missing <- setdiff(all_celltypes, names(celltype_colors))
if (length(missing) > 0) {
  cat("Warning: No colors defined for:", paste(missing, collapse = ", "), "\n")
} else {
  cat("All cell types have colors assigned\n")
}

## Step 3: Create Your First Sankey Plot

**Why start with a basic plot?** Before customizing, verify that your data loaded correctly and get an overview of the communication network. The default plot shows all interactions with sensible defaults.

In [ ]:
# Basic plot - default settings (flow_conserving, colored by sender, 600x900)
p <- sankey_plot(data, celltype_colors)
p

The plot displays four layers from left to right:
1. **Sender** (left) - Cell types sending signals
2. **Ligand** (center-left) - Signaling molecules produced by senders
3. **Receptor** (center-right) - Receptor molecules on receiver cells
4. **Receiver** (right) - Cell types receiving signals

Each flow traces one sender→ligand→receptor→receiver interaction. By default, flows are colored by **sender cell type**, and the plot uses **flow-conserving mode** (edge thickness reflects expression levels). Default dimensions are 600×900 pixels.

## Step 4: Customize Your Figure

ccflowR uses a `+` operator to layer customizations. Below are the most common adjustments for NicheNet data. See the **Parameter Reference Guide** for exhaustive visual examples of every parameter.

### Coloring Strategies

Color flows by `"sender"` (default), `"receiver"`, or `"ligand"` to answer different biological questions.

In [ ]:
# Color by receiver — "Who is receiving the most signals?"
p <- sankey_plot(data, celltype_colors) +
  sankey_colors(by = "receiver") +
  sankey_title("Colored by Receiver Cell Type")
p

### Flow Mode

The basic plot (Step 3) already uses **flow-conserving mode** by default — edge thickness reflects expression levels. Use `flow = "uniform"` if you only want to show network topology.

In [ ]:
# Uniform mode — all edges same thickness (network topology only)
p <- sankey_plot(data, celltype_colors, flow = "uniform") +
  sankey_title("Uniform Mode (Topology Only)")
p

### Highlighting Pathways

Use `sankey_highlight()` to focus attention on specific pathways while keeping the full network visible for context. You can highlight by `senders`, `receivers`, `ligands`, or `receptors` (multiple criteria use AND logic). See the **Parameter Reference Guide** for all highlighting options and `dim_alpha` variations.

In [ ]:
# Focus on signals FROM Macrophages
p <- sankey_plot(data, celltype_colors) +
  sankey_highlight(senders = "Macrophage", dim_alpha = 0.1) +
  sankey_title("Signals FROM Macrophages")
p

### Custom Node Ordering

Use `sankey_order()` to arrange nodes by biological narrative (e.g., differentiation stage, immune vs stromal). This is especially useful when generating multiple panels for side-by-side comparison.

In [ ]:
p <- sankey_plot(data, celltype_colors) +
  sankey_order(
    senders = c("Macrophage", "Fibroblast", "T_cell", "B_cell", "Endothelial"),
    receivers = c("T_cell", "B_cell", "Fibroblast", "Endothelial", "Macrophage")
  ) +
  sankey_title("Custom Node Order")
p

## Step 5: Export

Save your figure as HTML (interactive) or static formats (PNG, PDF, SVG). See the **Parameter Reference Guide** for all export options and format details.

In [ ]:
# Build a publication-ready figure
p <- sankey_plot(data, celltype_colors) +
  sankey_colors(by = "receiver") +
  sankey_highlight(senders = "Macrophage") +
  sankey_alpha(0.6) +
  sankey_title("Macrophage Signaling Network")

# Save as interactive HTML (always works, no dependencies)
sankey_save(p, "example_output.html")
cat("Saved HTML\n")

# Save as static PNG (requires kaleido or webshot2)
# sankey_save(p, "example_output.png", width = 700, height = 900)

# Clean up
if (file.exists("example_output.html")) file.remove("example_output.html")

## Complete Workflow

    library(ccflowR)

    # 1. Load your NicheNet/ContextNet output
    data <- read_nichenet("path/to/your/sankey_river.csv")
    summarize_data(data)

    # 2. Define cell type colors
    celltype_colors <- c(
      "B Cell" = "#17BECF", "T Cell" = "#006837",
      "Myeloid" = "#FF7F0E", "Fibroblast" = "#9C46D3",
      "Epithelial" = "#1F77B4", "Endothelial" = "#E89FBD"
    )

    # 3. Create and customize figure
    p <- sankey_plot(data, celltype_colors) +
      sankey_colors(by = "receiver") +
      sankey_highlight(senders = "Macrophage") +
      sankey_alpha(0.6) +
      sankey_title("Cell-Cell Communication")

    # 4. Save (see Parameter Reference Guide for all export options)
    sankey_save(p, "Figure1.html")
    sankey_save(p, "Figure1.png", width = 700, height = 900)

## Troubleshooting

| Issue | Solution |
|-------|----------|
| "Missing required columns" | Check column names: `sender`, `receiver`, `ligand`, `receptor` |
| "Falling back to uniform" | Add `scaled_activity` and `scaled_receptor_expression` columns |
| Plot not displaying | Re-run the cell (known htmlwidget issue) |
| Colors not applied | Ensure color vector names match cell type names exactly (case-sensitive) |

For general troubleshooting (export, labels, layout), see the **Parameter Reference Guide**.

## Next Steps
- **Multi-condition analysis**: See the [MultiNicheNet tutorial](tutorial-contextnet-multinichenet.ipynb)
- **All parameters**: See the [Parameter Reference Guide](tutorial-parameter-guide.ipynb) for visual examples of every customization option